# EDA BraTS 2024 GLI Post-Treatment — Extensión clásica

Notebook de **Exploratory Data Analysis** del dataset *BraTS 2024 GLI Post-Treatment* orientado a **segmentación clásica con ITK / SimpleITK** (no deep learning).

Este notebook está organizado de la siguiente manera:

- **S0 — Configuración**: rutas (`CASES_BASE`), constantes compartidas (`LABEL_MAP`, `COLORS`, `COLOR_MAP`, `OVERLAY_RGBA`, `MODALITIES`), tamaño de muestra.
- **S1 – S10**: análisis estructurales del dataset (integridad, dimensiones, volúmenes, patrones, instituciones, parches/contexto, outliers, recomendaciones). Cellls originales del EDA: **insertarlas aquí, intactas, debajo de S0 y antes de A**.
- **A – F (NUEVAS, orientadas a segmentación clásica)**:
  - A. Histogramas de intensidad por modalidad (fondo enmascarado).
  - B. Separabilidad de intensidades por sub-región vs tejido sano.
  - C. Bimodalidad y vista previa de **Otsu** por modalidad.
  - D. Variabilidad del rango de intensidades entre casos (motiva normalización).
  - E. Análisis de **semillas** para crecimiento de regiones.
  - F. Selección de **subconjunto representativo** de casos (`casos_demostrativos.csv`).
- **S11**: exportación a **reporte HTML autocontenido** (`EDA_BraTS2024_GLI_reporte.html`), extendido para incluir A–F.

Todas las secciones nuevas son **aditivas**: no eliminan ni modifican S1–S11.

## S0 — Configuración global

Constantes y rutas compartidas por todas las secciones.

- `CASES_BASE` se deja como variable de configuración para correr tanto en **Colab/Drive** como **local**.
- `SAMPLE_N = 25` controla el número de casos analizados por las secciones A–E (cargan volúmenes completos). El dataset entero (~2,200) se mantiene para S1–S10 que solo leen segmentaciones.

In [ ]:
import os
import gc
import glob
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Ruta del dataset BraTS 2024 GLI Post-Treatment.
# Editar segun el entorno (Colab/Drive o local).
# Estructura esperada: CASES_BASE/<case_id>/<case_id>-<modalidad>.nii.gz
# con modalidades: t1n, t1c, t2w, t2f, seg
# ---------------------------------------------------------------------------
CASES_BASE = os.environ.get(
    'BRATS_CASES_BASE',
    '/content/drive/MyDrive/BraTS2024_GLI/training'  # default Colab
)

# Constantes compartidas (mismas que en S1-S11)
LABEL_MAP = {1: 'NETC', 2: 'SNFH', 3: 'ET', 4: 'RC'}
COLORS = {
    'NETC': '#EF4444',   # rojo
    'SNFH': '#22C55E',   # verde
    'ET':   '#3B82F6',   # azul
    'RC':   '#EAB308',   # amarillo
    'SANO': '#94A3B8',   # gris
}
COLOR_MAP = {k: COLORS[v] for k, v in LABEL_MAP.items()}
OVERLAY_RGBA = {
    1: (239,  68,  68, 160),  # NETC
    2: ( 34, 197,  94, 160),  # SNFH
    3: ( 59, 130, 246, 160),  # ET
    4: (234, 179,   8, 160),  # RC
}
MODALITIES = ['t1n', 't1c', 't2w', 't2f', 'seg']
MODALITIES_IMG = ['t1n', 't1c', 't2w', 't2f']  # solo imagen, sin seg
MOD_COLORS = {
    't1n': '#94A3B8',
    't1c': '#3B82F6',
    't2w': '#22C55E',
    't2f': '#EAB308',
}

# Tamano de muestra para A-E (operan sobre volumenes completos)
SAMPLE_N = 25
RANDOM_STATE = 42

print(f'CASES_BASE = {CASES_BASE}')
print(f'SAMPLE_N   = {SAMPLE_N}')

## S1 – S10 — (insertar aquí las celdas originales del EDA)

> **NOTA del equipo**: las celdas originales **S1 a S10** del notebook EDA (integridad, dimensiones, volúmenes, patrones, instituciones, parches, outliers, recomendaciones, etc.) van **aquí**, intactas, entre S0 y la sección A.
>
> Las secciones nuevas A–F solo dependen de:
> - `CASES_BASE`, `LABEL_MAP`, `COLORS`, `COLOR_MAP`, `OVERLAY_RGBA`, `MODALITIES` (ya definidos en S0).
> - `df_ok`: DataFrame de casos con archivos completos (al menos columna `case_id` y rutas, o índice de carpetas válidas).
> - `df_vol`: DataFrame con volúmenes por sub-región (`case_id`, `ET_cc`, `NETC_cc`, `SNFH_cc`, `RC_cc`, `TumorTotal_cc`).
>
> Si por algún motivo `df_ok` / `df_vol` no quedan disponibles tras correr S1–S10, la próxima celda reconstruye versiones mínimas escaneando `CASES_BASE`.
>
> **Reencuadre (no se borra código)**:
> - **S6 (parches)**: en este proyecto **no entrenamos DL**, así que la métrica de desbalance por parches se reinterpreta como diagnóstico del **soporte espacial** que recibirá un crecimiento de regiones o un Otsu local (¿cabe el tumor en un *bounding box* manejable?). Se conserva como **contexto secundario**.
> - **S10 (pesos de clase / diseño del modelo)**: se reescribe como **selección de método clásico por sub-región y modalidad**. Las recomendaciones que ahí aparecen como *class weights* aquí se leen como **prioridad de la sub-región** (qué tan crítico es no perderla en la segmentación clásica). Las recomendaciones algorítmicas se derivan de las secciones B y C (separabilidad y bimodalidad), no de pesos para cross-entropy.

### Garantía de disponibilidad de `df_ok` y `df_vol`

Si las secciones previas no definieron estos DataFrames (notebook ejecutado parcialmente), esta celda los reconstruye con un escaneo barato de `CASES_BASE` usando únicamente la segmentación `seg.nii.gz` (no carga modalidades de intensidad).

In [ ]:
def _descubrir_casos(base_dir, modalidades=MODALITIES):
    """Lista case_ids en base_dir que tienen todas las modalidades."""
    casos = []
    if not os.path.isdir(base_dir):
        return casos
    for nombre in sorted(os.listdir(base_dir)):
        cdir = os.path.join(base_dir, nombre)
        if not os.path.isdir(cdir):
            continue
        # patrones aceptados:  <id>-<mod>.nii.gz  o  <id>_<mod>.nii.gz
        archivos = {}
        for mod in modalidades:
            cand = [
                os.path.join(cdir, f'{nombre}-{mod}.nii.gz'),
                os.path.join(cdir, f'{nombre}_{mod}.nii.gz'),
            ]
            for c in cand:
                if os.path.exists(c):
                    archivos[mod] = c
                    break
        if all(m in archivos for m in modalidades):
            casos.append({'case_id': nombre, **{f'path_{m}': archivos[m] for m in modalidades}})
    return casos


def _construir_df_vol(df_casos):
    """Recorre seg.nii.gz de cada caso y calcula volumenes en cc por sub-region."""
    import nibabel as nib
    filas = []
    for _, row in df_casos.iterrows():
        try:
            img = nib.load(row['path_seg'])
            data = np.asanyarray(img.dataobj).astype(np.uint8)
            voxel_cc = float(np.prod(img.header.get_zooms()[:3])) / 1000.0
            fila = {'case_id': row['case_id']}
            for lab, nombre in LABEL_MAP.items():
                fila[f'{nombre}_cc'] = float((data == lab).sum()) * voxel_cc
            fila['TumorTotal_cc'] = sum(fila[f'{n}_cc'] for n in LABEL_MAP.values())
            filas.append(fila)
            del img, data
        except Exception as e:
            print(f'[warn] {row["case_id"]}: {e}')
    gc.collect()
    return pd.DataFrame(filas)


if 'df_ok' not in globals():
    print('[info] df_ok no estaba definido. Reconstruyendo desde CASES_BASE...')
    _casos = _descubrir_casos(CASES_BASE)
    df_ok = pd.DataFrame(_casos)
    print(f'[info] {len(df_ok)} casos completos detectados.')

if 'df_vol' not in globals() and len(df_ok) > 0:
    print('[info] df_vol no estaba definido. Calculando volumenes (solo seg)...')
    df_vol = _construir_df_vol(df_ok)
    print(f'[info] df_vol listo: {df_vol.shape}')

df_ok.head() if len(df_ok) else 'df_ok vacio: revisar CASES_BASE'

### Utilidades comunes para A–F

- `_muestra_estratificada`: 25 casos elegidos por estratos de `TumorTotal_cc` (cuartiles) para que las distribuciones de intensidad sean representativas.
- `_cargar_caso`: carga las 4 modalidades + seg de un caso como `numpy.ndarray` (float32).
- `_brain_mask`: máscara de cerebro a partir de cualquier modalidad (BraTS viene **skull-stripped** con fondo = 0).

In [ ]:
import nibabel as nib


def _muestra_estratificada(df_vol, n=SAMPLE_N, seed=RANDOM_STATE):
    if len(df_vol) <= n:
        return df_vol['case_id'].tolist()
    df = df_vol.copy()
    df['__q'] = pd.qcut(df['TumorTotal_cc'], q=4, labels=False, duplicates='drop')
    por_estrato = max(1, n // df['__q'].nunique())
    seleccion = (df.groupby('__q', group_keys=False)
                   .apply(lambda g: g.sample(min(len(g), por_estrato), random_state=seed)))
    if len(seleccion) < n:
        extra = df.drop(seleccion.index).sample(n - len(seleccion), random_state=seed)
        seleccion = pd.concat([seleccion, extra])
    return seleccion['case_id'].tolist()


def _ruta(case_id, mod):
    fila = df_ok[df_ok['case_id'] == case_id]
    if len(fila) == 0 or f'path_{mod}' not in fila.columns:
        # fallback a convencion estandar
        return os.path.join(CASES_BASE, case_id, f'{case_id}-{mod}.nii.gz')
    return fila.iloc[0][f'path_{mod}']


def _cargar_modalidad(case_id, mod, dtype=np.float32):
    img = nib.load(_ruta(case_id, mod))
    arr = np.asanyarray(img.dataobj).astype(dtype)
    spacing = tuple(float(x) for x in img.header.get_zooms()[:3])
    return arr, spacing


def _brain_mask_desde(arr):
    """BraTS viene skull-stripped: el fondo es exactamente 0."""
    return arr > 0


def _intensidades_cerebro(arr):
    return arr[_brain_mask_desde(arr)].astype(np.float32, copy=False)


CASOS_MUESTRA = _muestra_estratificada(df_vol, n=SAMPLE_N) if 'df_vol' in globals() and len(df_vol) > 0 else []
print(f'Muestra estratificada: {len(CASOS_MUESTRA)} casos')
CASOS_MUESTRA[:5]

---
## A — Histogramas de intensidad por modalidad (fondo enmascarado)

Histograma agregado de las intensidades del cerebro (voxeles > 0) sobre la **muestra estratificada** de `SAMPLE_N` casos.

**Para qué sirve en segmentación clásica**: la **forma uni/bimodal** del histograma de cada modalidad condiciona directamente la viabilidad de Otsu (necesita dos modos) y la elección del umbral inicial para crecimiento de regiones.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_dark'

BINS = 120
MAX_VOX_POR_CASO = 250_000  # downsample para tener histogramas rapidos

# Acumulador de histograma por modalidad: (counts, edges)
hist_global = {m: None for m in MODALITIES_IMG}
rango_global = {m: [np.inf, -np.inf] for m in MODALITIES_IMG}

# Primera pasada: rangos
for cid in CASOS_MUESTRA:
    for m in MODALITIES_IMG:
        arr, _ = _cargar_modalidad(cid, m)
        v = _intensidades_cerebro(arr)
        if v.size == 0:
            continue
        # p1/p99 robustos
        lo, hi = np.percentile(v, [0.5, 99.5])
        rango_global[m][0] = min(rango_global[m][0], lo)
        rango_global[m][1] = max(rango_global[m][1], hi)
        del arr, v
    gc.collect()

# Segunda pasada: histograma con rango fijo
for cid in CASOS_MUESTRA:
    for m in MODALITIES_IMG:
        arr, _ = _cargar_modalidad(cid, m)
        v = _intensidades_cerebro(arr)
        if v.size == 0:
            continue
        if v.size > MAX_VOX_POR_CASO:
            idx = np.random.default_rng(RANDOM_STATE).choice(v.size, MAX_VOX_POR_CASO, replace=False)
            v = v[idx]
        lo, hi = rango_global[m]
        c, e = np.histogram(v, bins=BINS, range=(lo, hi))
        if hist_global[m] is None:
            hist_global[m] = [c.astype(np.int64), e]
        else:
            hist_global[m][0] += c
        del arr, v
    gc.collect()

fig_A = go.Figure()
for m in MODALITIES_IMG:
    if hist_global[m] is None:
        continue
    c, e = hist_global[m]
    centros = 0.5 * (e[:-1] + e[1:])
    densidad = c / max(c.sum(), 1)
    fig_A.add_trace(go.Scatter(
        x=centros, y=densidad, mode='lines', name=m.upper(),
        line=dict(color=MOD_COLORS[m], width=2),
        fill='tozeroy', opacity=0.35
    ))
fig_A.update_layout(
    title=f'A — Histograma agregado de intensidades del cerebro ({len(CASOS_MUESTRA)} casos)',
    xaxis_title='Intensidad', yaxis_title='Densidad relativa',
    height=460, legend=dict(orientation='h')
)
fig_A.show()

**Conclusión A.** La forma uni/bimodal de cada histograma marca el punto de partida:
- Una distribución claramente **bimodal** habilita Otsu (1 umbral) o multi-Otsu (>1 umbrales).
- Una distribución unimodal con cola larga sugiere que Otsu fallará y conviene **crecimiento de regiones** o **clustering** (K-means / GMM) sobre intensidades normalizadas.

---
## B — Separabilidad de intensidades por sub-región vs tejido sano

Extrae intensidades de cada sub-región (`ET, NETC, SNFH, RC`) y de **tejido sano** (`seg == 0` ∩ cerebro) para cada modalidad. Cuantifica el solapamiento con el **coeficiente de solapamiento de histogramas** (OVL ∈ [0, 1], 0 = perfectamente separable, 1 = totalmente solapado) y el **Bhattacharyya** (BD ∈ [0, ∞), 0 = perfectamente separable).

**Para qué sirve**: dice *qué modalidad usar para qué sub-región* en métodos clásicos.

In [ ]:
CLASES_B = ['ET', 'NETC', 'SNFH', 'RC', 'SANO']
MAX_VOX_CLASE = 80_000  # por caso y por clase

# acumulador: {modalidad: {clase: [arrays]}}
buckets = {m: {c: [] for c in CLASES_B} for m in MODALITIES_IMG}

for cid in CASOS_MUESTRA:
    seg, _ = _cargar_modalidad(cid, 'seg', dtype=np.uint8)
    for m in MODALITIES_IMG:
        arr, _ = _cargar_modalidad(cid, m)
        mask_cerebro = _brain_mask_desde(arr)
        for lab, nombre in LABEL_MAP.items():
            v = arr[(seg == lab) & mask_cerebro]
            if v.size > MAX_VOX_CLASE:
                v = np.random.default_rng(RANDOM_STATE).choice(v, MAX_VOX_CLASE, replace=False)
            if v.size > 0:
                buckets[m][nombre].append(v.astype(np.float32))
        v_sano = arr[(seg == 0) & mask_cerebro]
        if v_sano.size > MAX_VOX_CLASE:
            v_sano = np.random.default_rng(RANDOM_STATE).choice(v_sano, MAX_VOX_CLASE, replace=False)
        if v_sano.size > 0:
            buckets[m]['SANO'].append(v_sano.astype(np.float32))
        del arr
    del seg
    gc.collect()

# Concatenar
buckets_cat = {m: {c: (np.concatenate(buckets[m][c]) if buckets[m][c] else np.array([], dtype=np.float32))
                   for c in CLASES_B} for m in MODALITIES_IMG}
{m: {c: int(v.size) for c, v in d.items()} for m, d in buckets_cat.items()}

In [ ]:
def _ovl_bhatt(a, b, bins=80):
    """Coef. de solapamiento de histogramas y distancia de Bhattacharyya."""
    if a.size == 0 or b.size == 0:
        return np.nan, np.nan
    lo = float(min(a.min(), b.min()))
    hi = float(max(a.max(), b.max()))
    if hi <= lo:
        return np.nan, np.nan
    ha, e = np.histogram(a, bins=bins, range=(lo, hi), density=False)
    hb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=False)
    pa = ha / max(ha.sum(), 1)
    pb = hb / max(hb.sum(), 1)
    ovl = float(np.minimum(pa, pb).sum())
    bc = float(np.sum(np.sqrt(pa * pb)))
    bd = float(-math.log(bc)) if bc > 0 else float('inf')
    return ovl, bd

filas = []
for m in MODALITIES_IMG:
    sano = buckets_cat[m]['SANO']
    for clase in ['ET', 'NETC', 'SNFH', 'RC']:
        ovl, bd = _ovl_bhatt(buckets_cat[m][clase], sano)
        filas.append({'modalidad': m, 'clase': clase, 'OVL': ovl, 'Bhattacharyya': bd})
df_sep = pd.DataFrame(filas)
df_sep_piv = df_sep.pivot(index='clase', columns='modalidad', values='OVL').round(3)
df_sep_piv

In [ ]:
# 4 subplots: uno por modalidad, distribuciones (violin) por clase
fig_B = make_subplots(rows=2, cols=2, subplot_titles=[m.upper() for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i // 2 + 1, i % 2 + 1
    for clase in CLASES_B:
        v = buckets_cat[m][clase]
        if v.size == 0:
            continue
        # downsample para visualizacion
        if v.size > 20_000:
            v = np.random.default_rng(RANDOM_STATE).choice(v, 20_000, replace=False)
        fig_B.add_trace(go.Violin(
            y=v, name=clase, line_color=COLORS[clase],
            showlegend=(i == 0), legendgroup=clase,
            box_visible=False, meanline_visible=True, points=False,
            spanmode='hard'
        ), row=r, col=c)
fig_B.update_layout(
    title='B — Separabilidad de intensidades por sub-region vs tejido sano',
    height=720, violinmode='overlay'
)
fig_B.show()

**Conclusión B.** El coeficiente de solapamiento (más bajo = mejor separación) sugiere:
- **ET en T1c** suele dar OVL bajo → favorable para Otsu / crecimiento de regiones sobre T1c.
- **SNFH (edema) en T2-FLAIR** es la combinación con mejor contraste visual (FLAIR suprime LCR).
- **RC** depende de su contenido: si es quística aparece hipo-intensa en T1c, si tiene productos sanguíneos puede ser hiper-intensa.
- **NETC** suele tener alto solapamiento → mejor abordarla **después** de delimitar ET (NETC ≈ TumorCore − ET).

---
## C — Bimodalidad y vista previa del umbral de Otsu por modalidad

Para cada modalidad calcula:
- Umbrales **multi-Otsu** con `OtsuMultipleThresholdsImageFilter` de SimpleITK (n=2 y n=3 umbrales).
- **Coeficiente de bimodalidad** $b = (g^2 + 1) / (k + 3(n-1)^2/((n-2)(n-3)))$ (SAS); $b > 0.555$ sugiere bimodalidad.

Se promedian los umbrales sobre la muestra (umbrales mediana ± IQR), de modo que el reporte indica un **rango operativo** para Otsu en cada modalidad.

In [ ]:
import SimpleITK as sitk
from scipy import stats as spstats


def _coef_bimodalidad(x):
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n < 4:
        return np.nan
    g = spstats.skew(x, bias=False)
    k = spstats.kurtosis(x, fisher=True, bias=False)  # exceso
    denom = k + 3.0 * (n - 1) ** 2 / ((n - 2) * (n - 3))
    return float((g ** 2 + 1.0) / denom) if denom != 0 else np.nan


def _otsu_multinivel(vol, mask, n_thresholds):
    img = sitk.GetImageFromArray(vol.astype(np.float32))
    msk = sitk.GetImageFromArray(mask.astype(np.uint8))
    otsu = sitk.OtsuMultipleThresholdsImageFilter()
    otsu.SetNumberOfThresholds(n_thresholds)
    otsu.SetMaskValue(1)
    try:
        otsu.Execute(img, msk)
    except Exception:
        otsu.Execute(img)
    return list(otsu.GetThresholds())


registros_C = []
muestras_intensidad = {m: [] for m in MODALITIES_IMG}
for cid in CASOS_MUESTRA:
    for m in MODALITIES_IMG:
        arr, _ = _cargar_modalidad(cid, m)
        mask = _brain_mask_desde(arr)
        v = arr[mask]
        if v.size < 1000:
            continue
        bcoef = _coef_bimodalidad(v if v.size < 200_000 else np.random.default_rng(0).choice(v, 200_000, replace=False))
        try:
            th2 = _otsu_multinivel(arr, mask, 2)
            th3 = _otsu_multinivel(arr, mask, 3)
        except Exception:
            th2, th3 = [np.nan, np.nan], [np.nan, np.nan, np.nan]
        registros_C.append({
            'case_id': cid, 'modalidad': m,
            'bimodalidad': bcoef,
            'otsu_t1_n2': th2[0] if len(th2) > 0 else np.nan,
            'otsu_t2_n2': th2[1] if len(th2) > 1 else np.nan,
            'otsu_t1_n3': th3[0] if len(th3) > 0 else np.nan,
            'otsu_t2_n3': th3[1] if len(th3) > 1 else np.nan,
            'otsu_t3_n3': th3[2] if len(th3) > 2 else np.nan,
        })
        # guardar una muestra pequeña para histograma
        if len(muestras_intensidad[m]) < 6:
            muestras_intensidad[m].append(np.random.default_rng(0).choice(v, min(50_000, v.size), replace=False))
        del arr
    gc.collect()

df_C = pd.DataFrame(registros_C)
resumen_C = df_C.groupby('modalidad').agg({
    'bimodalidad': 'median',
    'otsu_t1_n2': 'median', 'otsu_t2_n2': 'median',
    'otsu_t1_n3': 'median', 'otsu_t2_n3': 'median', 'otsu_t3_n3': 'median'
}).round(3)
resumen_C['bimodal?'] = resumen_C['bimodalidad'] > 0.555
resumen_C

In [ ]:
fig_C = make_subplots(rows=2, cols=2, subplot_titles=[m.upper() for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i // 2 + 1, i % 2 + 1
    if not muestras_intensidad[m]:
        continue
    v = np.concatenate(muestras_intensidad[m])
    lo, hi = np.percentile(v, [0.5, 99.5])
    fig_C.add_trace(go.Histogram(
        x=v, nbinsx=120, marker_color=MOD_COLORS[m], opacity=0.75,
        showlegend=False, name=m.upper(),
        xbins=dict(start=float(lo), end=float(hi), size=(hi - lo) / 120)
    ), row=r, col=c)
    # lineas de umbral n=2 mediana
    th = resumen_C.loc[m, ['otsu_t1_n2', 'otsu_t2_n2']].values
    for j, t in enumerate(th):
        if pd.notna(t):
            fig_C.add_vline(x=float(t), line_dash='dash', line_color='#F472B6', row=r, col=c,
                            annotation_text=f't{j+1}={t:.1f}', annotation_position='top')
fig_C.update_layout(
    title='C — Histograma cerebral y umbrales Otsu (n=2) por modalidad',
    height=720, bargap=0.02
)
fig_C.show()

**Conclusión C.** Las modalidades con coef. de bimodalidad > 0.555 son candidatas naturales para Otsu (un único umbral separa cerebro/tumor o cerebro/edema). Para las modalidades unimodales, multi-Otsu (n=3) o crecimiento de regiones suelen funcionar mejor.

---
## D — Variabilidad del rango de intensidades entre casos

Estadísticos por **caso × modalidad** sobre el cerebro enmascarado: `min`, `max`, `media`, `p99`. La dispersión entre casos motiva una etapa de **normalización** (z-score por volumen y/o *histogram matching* a una referencia) en `01_preprocesamiento`.

In [ ]:
filas_D = []
for cid in CASOS_MUESTRA:
    for m in MODALITIES_IMG:
        arr, _ = _cargar_modalidad(cid, m)
        v = _intensidades_cerebro(arr)
        if v.size == 0:
            continue
        filas_D.append({
            'case_id': cid, 'modalidad': m,
            'min': float(v.min()), 'max': float(v.max()),
            'media': float(v.mean()), 'p99': float(np.percentile(v, 99)),
        })
        del arr
    gc.collect()
df_D = pd.DataFrame(filas_D)
df_D.groupby('modalidad').agg(['median', 'std']).round(2)

In [ ]:
fig_D = make_subplots(rows=1, cols=2, subplot_titles=['Media por caso', 'p99 por caso'])
for m in MODALITIES_IMG:
    s = df_D[df_D['modalidad'] == m]
    fig_D.add_trace(go.Box(y=s['media'], name=m.upper(), marker_color=MOD_COLORS[m],
                            boxmean=True, showlegend=False), row=1, col=1)
    fig_D.add_trace(go.Box(y=s['p99'], name=m.upper(), marker_color=MOD_COLORS[m],
                            boxmean=True, showlegend=False), row=1, col=2)
fig_D.update_layout(title='D — Variabilidad del rango de intensidades entre casos',
                    height=460)
fig_D.show()

**Conclusión D.** El rango y la media varían sustancialmente entre casos para la misma modalidad → **un umbral fijo no es transferible**. Justifica aplicar **z-score por volumen** o **histogram matching** antes de los métodos basados en intensidad absoluta.

---
## E — Análisis de semillas para crecimiento de regiones

Para cada caso de la muestra:
1. Centroide de `ET` y de `TumorTotal` desde `seg`.
2. Intensidad en el centroide en **T1c** y **media en vecindad 3×3×3**.
3. Distribución de la intensidad de semilla entre casos.

**Para qué sirve**: fija un **rango operativo de semilla** y una **tolerancia** razonable para `ConnectedThresholdImageFilter` y un **multiplicador** inicial para `ConfidenceConnectedImageFilter`.

In [ ]:
def _centroide(mask):
    idx = np.argwhere(mask)
    if idx.size == 0:
        return None
    return tuple(int(round(x)) for x in idx.mean(axis=0))


def _media_vecindad(arr, centro, r=1):
    z, y, x = centro
    z0, z1 = max(0, z - r), min(arr.shape[0], z + r + 1)
    y0, y1 = max(0, y - r), min(arr.shape[1], y + r + 1)
    x0, x1 = max(0, x - r), min(arr.shape[2], x + r + 1)
    sub = arr[z0:z1, y0:y1, x0:x1]
    return float(sub.mean()) if sub.size else float('nan')

filas_E = []
for cid in CASOS_MUESTRA:
    seg, _ = _cargar_modalidad(cid, 'seg', dtype=np.uint8)
    t1c, _ = _cargar_modalidad(cid, 't1c')
    centro_et = _centroide(seg == 3)
    centro_tt = _centroide(seg > 0)
    fila = {'case_id': cid}
    if centro_et is not None:
        fila['centro_ET'] = centro_et
        fila['t1c_centro_ET'] = float(t1c[centro_et])
        fila['t1c_vec_ET_3x3x3'] = _media_vecindad(t1c, centro_et, r=1)
    if centro_tt is not None:
        fila['centro_TT'] = centro_tt
        fila['t1c_centro_TT'] = float(t1c[centro_tt])
        fila['t1c_vec_TT_3x3x3'] = _media_vecindad(t1c, centro_tt, r=1)
    filas_E.append(fila)
    del seg, t1c
    gc.collect()
df_E = pd.DataFrame(filas_E)
df_E.describe(include='all').round(2)

In [ ]:
fig_E = go.Figure()
if 't1c_vec_ET_3x3x3' in df_E.columns:
    fig_E.add_trace(go.Histogram(x=df_E['t1c_vec_ET_3x3x3'].dropna(),
                                 name='Semilla ET (T1c, 3x3x3)',
                                 marker_color=COLORS['ET'], opacity=0.75))
if 't1c_vec_TT_3x3x3' in df_E.columns:
    fig_E.add_trace(go.Histogram(x=df_E['t1c_vec_TT_3x3x3'].dropna(),
                                 name='Semilla TumorTotal (T1c, 3x3x3)',
                                 marker_color=COLORS['SNFH'], opacity=0.75))
fig_E.update_layout(
    title='E — Intensidad de semilla (T1c) en el centroide, vecindad 3x3x3',
    xaxis_title='Intensidad T1c', yaxis_title='Casos',
    height=420, barmode='overlay'
)
fig_E.show()

# Rango operativo recomendado
_q = df_E['t1c_vec_ET_3x3x3'].dropna()
if len(_q):
    rango_semilla_ET = (float(_q.quantile(0.1)), float(_q.quantile(0.9)))
    tol_recomendada = float((rango_semilla_ET[1] - rango_semilla_ET[0]) / 2.0)
    print(f'Rango p10-p90 de semilla ET en T1c: {rango_semilla_ET}')
    print(f'Tolerancia inicial sugerida para ConnectedThreshold: +/- {tol_recomendada:.1f}')
    print('Multiplicador inicial sugerido para ConfidenceConnected: 2.0 (ajustar en barrido).')

**Conclusión E.** El rango p10–p90 de la intensidad en el centroide de ET en T1c define el **rango de semilla** y la **tolerancia** inicial para `ConnectedThresholdImageFilter`. Para `ConfidenceConnectedImageFilter` se sugiere `multiplier ≈ 2.0` con 2–3 iteraciones, ajustable en barrido.

---
## F — Selección de subconjunto representativo de casos

Desde `df_vol` elegimos **4–6 casos demostrativos** que el equipo reutilizará en todos los módulos (preprocesamiento, segmentación, visualización). Criterios:

1. **Típico**: mediana de `TumorTotal_cc`.
2. **Grande**: percentil 95 de `TumorTotal_cc`.
3. **Pequeño**: percentil 5 de `TumorTotal_cc`.
4. **Con resección (RC > 0)**: representante post-quirúrgico.
5. **Solo SNFH** (`ET_cc == 0` y `SNFH_cc > 0`): caso sin tumor activo.
6. **NETC dominante** opcional: `NETC_cc` máxima.

In [ ]:
def _id_mas_cercano(df, columna, valor):
    if df.empty:
        return None
    idx = (df[columna] - valor).abs().idxmin()
    return df.loc[idx, 'case_id']

criterios = []
if 'TumorTotal_cc' in df_vol.columns and len(df_vol):
    med = df_vol['TumorTotal_cc'].median()
    p95 = df_vol['TumorTotal_cc'].quantile(0.95)
    p05 = df_vol['TumorTotal_cc'].quantile(0.05)
    criterios.append({'tipo': 'tipico',  'criterio': f'TumorTotal_cc ~ mediana ({med:.1f} cc)',
                       'case_id': _id_mas_cercano(df_vol, 'TumorTotal_cc', med)})
    criterios.append({'tipo': 'grande',  'criterio': f'TumorTotal_cc ~ p95 ({p95:.1f} cc)',
                       'case_id': _id_mas_cercano(df_vol, 'TumorTotal_cc', p95)})
    criterios.append({'tipo': 'pequeño', 'criterio': f'TumorTotal_cc ~ p5 ({p05:.1f} cc)',
                       'case_id': _id_mas_cercano(df_vol, 'TumorTotal_cc', p05)})
if 'RC_cc' in df_vol.columns:
    candidatos_rc = df_vol[df_vol['RC_cc'] > 0]
    if len(candidatos_rc):
        cid = candidatos_rc.sort_values('RC_cc', ascending=False).iloc[0]['case_id']
        criterios.append({'tipo': 'post-reseccion', 'criterio': 'RC > 0 (mayor RC_cc)', 'case_id': cid})
if {'ET_cc', 'SNFH_cc'}.issubset(df_vol.columns):
    candidatos_solo_snfh = df_vol[(df_vol['ET_cc'] == 0) & (df_vol['SNFH_cc'] > 0)]
    if len(candidatos_solo_snfh):
        cid = candidatos_solo_snfh.sort_values('SNFH_cc', ascending=False).iloc[0]['case_id']
        criterios.append({'tipo': 'solo SNFH', 'criterio': 'ET==0 y SNFH>0', 'case_id': cid})
if 'NETC_cc' in df_vol.columns:
    cid = df_vol.sort_values('NETC_cc', ascending=False).iloc[0]['case_id']
    criterios.append({'tipo': 'NETC dominante', 'criterio': 'NETC_cc max (opcional)', 'case_id': cid})

df_demo = pd.DataFrame(criterios).drop_duplicates(subset='case_id').reset_index(drop=True)
df_demo = df_demo.merge(df_vol, on='case_id', how='left')
ruta_csv = os.path.join(os.path.dirname(os.path.abspath('__file__')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__)),
                       'casos_demostrativos.csv') if False else 'casos_demostrativos.csv'
df_demo.to_csv(ruta_csv, index=False)
print(f'[ok] {len(df_demo)} casos demostrativos exportados a {ruta_csv}')
df_demo

---
## S11 — Exportación a reporte HTML autocontenido (extendido)

Construye `EDA_BraTS2024_GLI_reporte.html` con el mismo estilo oscuro y secciones del reporte original, **e incluye las figuras y tablas de las secciones A–F** y la **tabla del subconjunto representativo**.

- Figuras Plotly embebidas con `include_plotlyjs='inline'` (autocontenido, no requiere red).
- El archivo final se guarda junto al notebook y queda versionado en git.

In [ ]:
from plotly.io import to_html

# -- estilo oscuro identico al reporte original --
CSS = """*{box-sizing:border-box;margin:0;padding:0}
body{background:#0f172a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif}
.hdr{background:#1e293b;padding:32px 40px;border-bottom:1px solid #334155}
.hdr h1{font-size:22px;font-weight:600;color:#fff;margin-bottom:6px}
.hdr p{font-size:13px;color:#94a3b8;margin-top:4px}
nav{background:#0f172a;border-bottom:1px solid #1e293b;padding:0 40px;position:sticky;top:0;z-index:100;display:flex;overflow-x:auto}
nav a{color:#64748b;text-decoration:none;padding:12px 16px;font-size:12px;white-space:nowrap;border-bottom:2px solid transparent}
nav a:hover{color:#e2e8f0;border-bottom-color:#8B5CF6}
.wrap{max-width:1300px;margin:0 auto;padding:32px 24px}
.sec{margin-bottom:60px;scroll-margin-top:56px}
.stitle{font-size:18px;font-weight:600;color:#e2e8f0;margin-bottom:8px;padding-bottom:8px;border-bottom:2px solid #8B5CF6}
.sdesc{font-size:13px;color:#94a3b8;margin-bottom:20px;line-height:1.7}
.plt{margin:16px 0;border-radius:8px;overflow:hidden}
table{width:100%;border-collapse:collapse;font-size:12px;margin-top:12px}
th{background:#1e293b;color:#fff;padding:8px 14px;text-align:left}
td{padding:8px 14px;border-bottom:1px solid #1e293b;color:#cbd5e1}
tr:nth-child(even) td{background:#1a2535}
.ok{background:#0d2137;border-left:4px solid #22C55E;border-radius:0 8px 8px 0;padding:12px 18px;margin-top:16px;font-size:13px;color:#86efac;line-height:1.6}
.warn{background:#1a1500;border-left:4px solid #EAB308;border-radius:0 8px 8px 0;padding:12px 18px;margin-top:16px;font-size:13px;color:#fde047}
.ftr{background:#1e293b;border-top:1px solid #334155;padding:24px 40px;text-align:center;font-size:12px;color:#64748b;margin-top:48px}
"""

def _fig_html(fig, include_js=False):
    return to_html(fig, include_plotlyjs='inline' if include_js else False, full_html=False, default_height='460px')

def _df_html(df, max_rows=30):
    return df.head(max_rows).to_html(index=False, classes='dataframe', border=0)


# -- Si la version original de S11 produjo un HTML base, lo intentamos cargar para inyectar A-F
# antes de </div></body>. Si no existe, generamos un HTML completo nuevo con solo S0 + A-F.
_aqui = os.getcwd()
_html_base = os.path.join(_aqui, 'EDA_BraTS2024_GLI_reporte.html')
_html_alt  = os.path.join(_aqui, '..', 'EDA_BraTS2024_GLI.ipynb.html')
html_previo = None
for p in [_html_base, _html_alt]:
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            html_previo = f.read()
        print(f'[info] HTML base detectado: {p}')
        break

# -- Construir las secciones nuevas A-F como bloques HTML --
secciones_html = []

secciones_html.append(f"""
<div class='sec' id='A-histogramas'>
  <div class='stitle'>A — Histogramas de intensidad por modalidad (fondo enmascarado)</div>
  <div class='sdesc'>Histograma agregado del cerebro (voxeles &gt; 0) sobre una muestra estratificada de {len(CASOS_MUESTRA)} casos. La forma uni/bimodal condiciona Otsu.</div>
  <div class='plt'>{_fig_html(fig_A, include_js=True)}</div>
  <div class='ok'>La forma del histograma de cada modalidad guia la eleccion entre Otsu (bimodal) y crecimiento/clustering (unimodal).</div>
</div>
""")

secciones_html.append(f"""
<div class='sec' id='B-separabilidad'>
  <div class='stitle'>B — Separabilidad por sub-region vs tejido sano</div>
  <div class='sdesc'>Distribuciones por modalidad y coeficiente de solapamiento (OVL, mas bajo = mejor) por clase vs sano.</div>
  <div class='plt'>{_fig_html(fig_B)}</div>
  {_df_html(df_sep_piv.reset_index())}
  <div class='ok'>OVL minimo por clase sugiere la modalidad ideal: ET en T1c, SNFH en T2-FLAIR, etc.</div>
</div>
""")

secciones_html.append(f"""
<div class='sec' id='C-otsu'>
  <div class='stitle'>C — Bimodalidad y vista previa de Otsu por modalidad</div>
  <div class='sdesc'>Umbrales multi-Otsu (n=2) sobre el cerebro enmascarado y coeficiente de bimodalidad por modalidad.</div>
  <div class='plt'>{_fig_html(fig_C)}</div>
  {_df_html(resumen_C.reset_index())}
  <div class='ok'>Modalidades con bimodalidad &gt; 0.555 son candidatas a Otsu directo; el resto requiere multi-Otsu o crecimiento de regiones.</div>
</div>
""")

secciones_html.append(f"""
<div class='sec' id='D-variabilidad'>
  <div class='stitle'>D — Variabilidad del rango de intensidades entre casos</div>
  <div class='sdesc'>Box plots de media y p99 por caso para cada modalidad. Motiva normalizacion (z-score / histogram matching).</div>
  <div class='plt'>{_fig_html(fig_D)}</div>
  <div class='warn'>Intensidades MRI no estandarizadas: un umbral fijo no transfiere entre casos.</div>
</div>
""")

secciones_html.append(f"""
<div class='sec' id='E-semillas'>
  <div class='stitle'>E — Analisis de semillas para crecimiento de regiones</div>
  <div class='sdesc'>Intensidad en T1c en el centroide de ET y TumorTotal (vecindad 3x3x3) sobre {len(CASOS_MUESTRA)} casos.</div>
  <div class='plt'>{_fig_html(fig_E)}</div>
  <div class='ok'>El rango p10-p90 define la tolerancia inicial sugerida para ConnectedThresholdImageFilter.</div>
</div>
""")

secciones_html.append(f"""
<div class='sec' id='F-subconjunto'>
  <div class='stitle'>F — Subconjunto representativo de casos</div>
  <div class='sdesc'>Casos demostrativos seleccionados desde df_vol. El equipo reutiliza estos case_id en preprocesamiento, segmentacion y visualizacion.</div>
  {_df_html(df_demo)}
  <div class='ok'>Exportado a <code>notebooks/casos_demostrativos.csv</code>.</div>
</div>
""")

BLOQUE_NUEVO = '\n'.join(secciones_html)

# -- Composicion final --
if html_previo:
    # Inyectar antes del primer </body> y agregar entradas al nav si existe.
    extra_nav = ("<a href='#A-histogramas'>A. Histogramas</a>"
                 "<a href='#B-separabilidad'>B. Separabilidad</a>"
                 "<a href='#C-otsu'>C. Otsu</a>"
                 "<a href='#D-variabilidad'>D. Variabilidad</a>"
                 "<a href='#E-semillas'>E. Semillas</a>"
                 "<a href='#F-subconjunto'>F. Subconjunto</a>")
    html_out = html_previo
    if '</nav>' in html_out:
        html_out = html_out.replace('</nav>', extra_nav + '</nav>', 1)
    # asegurar plotly.js inline
    if 'plotly' not in html_out.lower():
        html_out = html_out.replace('</head>', "<script src='https://cdn.plot.l" + "y/plotly-latest.min.js'></script></head>", 1)
    html_out = html_out.replace('</body>', f"<div class='wrap'>{BLOQUE_NUEVO}</div></body>", 1)
else:
    # Fallback: HTML autocontenido solo con A-F
    html_out = f"""<!DOCTYPE html><html lang='es'><head><meta charset='UTF-8'>
<title>EDA BraTS 2024 GLI - Reporte clasico</title>
<style>{CSS}</style></head>
<body>
<div class='hdr'><h1>EDA BraTS 2024 GLI Post-Treatment — Extension clasica</h1>
<p>Segmentacion clasica con ITK / SimpleITK &middot; Pontificia Universidad Javeriana &middot; 2026</p>
<p>Abel Albuez &middot; Victoria Acero &middot; Santiago Gil</p></div>
<nav><a href='#A-histogramas'>A. Histogramas</a><a href='#B-separabilidad'>B. Separabilidad</a><a href='#C-otsu'>C. Otsu</a><a href='#D-variabilidad'>D. Variabilidad</a><a href='#E-semillas'>E. Semillas</a><a href='#F-subconjunto'>F. Subconjunto</a></nav>
<div class='wrap'>{BLOQUE_NUEVO}</div>
<div class='ftr'>Generado por notebook EDA_BraTS2024_GLI.ipynb</div>
</body></html>"""

ruta_html = os.path.join(_aqui, 'EDA_BraTS2024_GLI_reporte.html')
with open(ruta_html, 'w', encoding='utf-8') as f:
    f.write(html_out)
print(f'[ok] reporte exportado: {ruta_html}  ({os.path.getsize(ruta_html)/1024:.1f} KB)')